# StructflyDocTruth Flow Testing

This notebook loads sample documents from `notebooks/data`, configures DSPy using the current environment settings, and runs the LangGraph flow directly.

In [1]:
from pathlib import Path
import json
import sys
from pprint import pprint

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
APP_ROOT = PROJECT_ROOT / "StructflyDocTruth"

for candidate in (str(PROJECT_ROOT), str(APP_ROOT)):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

DATA_DIR = NOTEBOOK_DIR / "data"
sorted(path.name for path in DATA_DIR.glob("*.txt"))

['contract_sample.txt', 'invoice_sample.txt', 'resume_sample.txt']

In [2]:
from app.core.dspy_config import configure_dspy
from app.core.settings import get_settings
from app.graph.workflow import graph

settings = get_settings()
lm = configure_dspy(settings)

print({
    "dspy_model": settings.dspy_model_identifier,
    "dspy_api_base": settings.dspy_api_base,
    "default_source_type": settings.default_source_type,
})

/Users/rishikeeshmahamuni/structfly-doc-truth/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


{'dspy_model': 'ollama_chat/qwen2.5:7b', 'dspy_api_base': 'http://localhost:11434', 'default_source_type': 'text'}


In [3]:
def load_document(name: str) -> str:
    return (DATA_DIR / name).read_text(encoding="utf-8")

sample_name = "invoice_sample.txt"
sample_text = load_document(sample_name)
print(sample_text)

Invoice Number: INV-1001
Vendor: ABC Ltd
Customer: Structfly Technologies
Invoice Date: 2026-04-01
Total Amount: 12500 INR



In [4]:
def run_flow(document_name: str, source_type: str = "text") -> dict:
    text = load_document(document_name)
    return graph.invoke({"raw_text": text, "source_type": source_type})

result = run_flow(sample_name)
pprint(result)

{'consolidated_fields': [{'proposed_name': 'invoice_number',
                          'raw_value': 'INV-1001'},
                         {'proposed_name': 'vendor', 'raw_value': 'ABC Ltd'},
                         {'proposed_name': 'customer',
                          'raw_value': 'Structfly Technologies'},
                         {'proposed_name': 'invoice_date',
                          'raw_value': '2026-04-01'},
                         {'proposed_name': 'total_amount',
                          'raw_value': '12500 INR'}],
 'document_id': 'doc_61e7e6a871b649f1bd33545f37863dec',
 'document_type_guess': 'Invoice',
 'extracted_text': 'Invoice Number: INV-1001\n'
                   'Vendor: ABC Ltd\n'
                   'Customer: Structfly Technologies\n'
                   'Invoice Date: 2026-04-01\n'
                   'Total Amount: 12500 INR',
 'ground_truth_record': {'document_id': 'doc_61e7e6a871b649f1bd33545f37863dec',
                         'fields': [{'proposed_name': 

In [ ]:
all_results = {}

for doc_path in sorted(DATA_DIR.glob("*.txt")):
    all_results[doc_path.name] = run_flow(doc_path.name)

print(json.dumps(all_results, indent=2, default=str))